# Лабораторная работа №1. K ближайших соседей

Фокин Леонид Александрович, группа 401, вариант 20.

## Данные и метрики

**Wholesale Customers.** По структуре годовых закупок можно определить, обслуживает ли оптовик заведение Horeca или розничный магазин, например при переназначении клиентов между менеджерами. Цель `Channel`: 1 = Horeca, 2 = Retail. Классы неравны; macro-F1 даёт каждому одинаковый вес, accuracy оставляю для сравнения доли верных ответов.

**SGEMM GPU Kernel Performance.** Хочу оценить время матричного умножения для ещё не проверенных параметров одного GPU-ядра, чтобы сократить перебор замеров. Цель: среднее четырёх прогонов в мс. MAE даёт типичный масштаб абсолютной ошибки в тех же единицах, RMSE сильнее штрафует большие промахи. Случайный тест проверяет новые конфигурации из той же совокупности, а не перенос на другое железо.

Тест: 20%, `random_state=20`, для классификации со стратификацией. Во всех пяти работах используются одни и те же строки. CV и первые графики используют только обучающую часть. Источники и отбор строк: [data/README.md](data/README.md).

Загружаю общие функции разбиения и оценки. В этой работе библиотечный бейзлайн использует исходные числовые признаки.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
import numpy as np
import pandas as pd
from IPython.display import display
from study import SEED, TASKS, evaluate, split, validate

BASE = 'raw'

def sklearn_model(task, params):
    if task == 'classification':
        return KNeighborsClassifier(**params)
    return KNeighborsRegressor(**params)

### Распределения в train

Проверяю дисбаланс классов и правый хвост времени выполнения. Логарифмическая шкала оси X не меняет целевую переменную модели.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier, DummyRegressor

_, _, labels, _ = split('classification')
_, _, runtimes, _ = split('regression')
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3), layout='constrained')
labels.value_counts().rename(index={1: 'Horeca', 2: 'Retail'}).plot.bar(ax=a, rot=0, color='#35688b')
a.set(ylabel='Покупатели в train')
b.hist(runtimes, bins=np.geomspace(runtimes.min(), runtimes.max(), 25), color='#a85c34')
b.set(xscale='log', xlabel='Время, мс (лог. шкала)', ylabel='Конфигурации в train')
plt.show()
display(pd.Series({task: len(split(task)[0]) for task in TASKS}, name='train'))

В качестве нижней точки отсчёта: самый частый канал и медианное время. Эти модели не участвуют в подборе параметров.

In [ ]:
dummy = [evaluate('classification', DummyClassifier(strategy='most_frequent'))[0],
         evaluate('regression', DummyRegressor(strategy='median'))[0]]
display(pd.DataFrame(dummy).drop(columns='fit_seconds').round(4))

## Бейзлайн sklearn

Фиксирую качество моделей со стандартными параметрами. Тестовые оценки не участвуют в выборе конфигурации.

In [ ]:
baseline = {task: evaluate(task, sklearn_model(task, {}), BASE) for task in TASKS}
display(pd.DataFrame([metrics for metrics, _ in baseline.values()]).round(4))